# Modelos COMPAS (réplica del paper)

<!-- Este notebook:
- Carga los CSV generados en el notebook de preprocesamiento.
- Entrena los modelos del paper:
  - Logistic Regression
  - SVM
  - XGBoost
- Usa los conjuntos de 2, 7 y 8 variables.
- Evalúa cada modelo con validación cruzada estratificada de 10 folds.
- Calcula métricas:
  - Accuracy
  - Precision
  - Recall
  - F1
  - Conteos de la matriz de confusión
- Guarda resultados agregados y por fold en un archivo JSON.
- Genera CSV por modelo con predicciones out-of-fold para auditoría con Aequitas. -->


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from xgboost import XGBClassifier

## 1. Cargar datos preprocesados y contexto de auditoría


In [3]:
# Se usa la misma ruta relativa que el notebook de preprocesamiento.
# Se conserva la ruta anterior como respaldo para no romper ejecuciones existentes.
BASE_DIR = Path("data") / "processed"
LEGACY_BASE_DIR = Path("Evaluacion-de-librerias-para-Inteligencia-Artificial-responsable/data/processed")
if not BASE_DIR.exists() and LEGACY_BASE_DIR.exists():
    BASE_DIR = LEGACY_BASE_DIR

# df2 = pd.read_csv(BASE_DIR / "compas_features_2.csv")  # 2 características
# df7 = pd.read_csv(BASE_DIR / "compas_features_7.csv")  # 7 características
df8 = pd.read_csv(BASE_DIR / "compas_preprocessed.csv")  # 8 características

audit_context = pd.read_csv(BASE_DIR / "compas_audit_context.csv")

TARGET = "two_year_recid"
REQUIRED_AUDIT_COLUMNS = {"entity_id", "race", "sex", "age_cat", "label_value"}
missing_audit_columns = REQUIRED_AUDIT_COLUMNS.difference(audit_context.columns)
if missing_audit_columns:
    raise ValueError(
        "El contexto de auditoría no contiene las columnas requeridas: "
        f"{sorted(missing_audit_columns)}"
    )


def validate_audit_alignment(model_df: pd.DataFrame, dataset_name: str) -> None:
    """Verifica la alineación fila a fila entre datos de modelo y contexto Aequitas."""
    if len(model_df) != len(audit_context):
        raise ValueError(
            f"{dataset_name} y compas_audit_context.csv tienen distinto número de filas."
        )
    if not np.array_equal(
        model_df[TARGET].to_numpy(), audit_context["label_value"].to_numpy()
    ):
        raise ValueError(
            f"Las etiquetas de {dataset_name} no coinciden con compas_audit_context.csv."
        )


for dataset_name, model_df in {
    # "compas_features_2.csv": df2,
    # "compas_features_7.csv": df7,
    "compas_preprocessed.csv": df8,
}.items():
    validate_audit_alignment(model_df, dataset_name)

print(df8.shape)
print("Contexto de auditoría:", audit_context.shape)


(6172, 8)
Contexto de auditoría: (6172, 5)


## 2. Funciones de evaluación

In [4]:
def compute_metrics(y_true, y_pred):
    # labels=[0, 1] evita errores si, excepcionalmente, algún fold no contiene
    # predicciones de una de las clases.
    accuracy = accuracy_score(y_true, y_pred)
    precision = np.nan_to_num(
        confusion_matrix(y_true, y_pred, labels=[0, 1])[1, 1]
        / (confusion_matrix(y_true, y_pred, labels=[0, 1])[1, 1] + confusion_matrix(y_true, y_pred, labels=[0, 1])[0, 1])
    )
    recall = np.nan_to_num(
        confusion_matrix(y_true, y_pred, labels=[0, 1])[1, 1]
        / (confusion_matrix(y_true, y_pred, labels=[0, 1])[1, 1] + confusion_matrix(y_true, y_pred, labels=[0, 1])[1, 0])
    )
    f1 = np.nan_to_num(
        2 * precision * recall / (precision + recall)
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
    }

## 3. Función de entrenamiento con validación cruzada estratificada

<!-- Se usa `StratifiedKFold` con 10 folds para conservar aproximadamente la misma proporción de clases de `two_year_recid` en cada partición. En cada fold se entrena una copia nueva del modelo y se evalúa sobre el fold reservado.

Además de las métricas agregadas, se guardan predicciones *out-of-fold* por persona. Cada archivo de auditoría conserva `entity_id`, grupos protegidos sin codificar, `label_value` y `score`, que son las columnas necesarias para Aequitas. -->


In [5]:
def _to_python_number(value):
    """Convierte escalares de NumPy a tipos serializables por JSON."""
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value


def aggregate_metrics(fold_metrics):
    """Calcula el promedio y la desviación estándar de las métricas de los folds."""
    metric_names = fold_metrics[0].keys()

    mean_metrics = {
        metric: _to_python_number(np.mean([fold[metric] for fold in fold_metrics]))
        for metric in metric_names
    }
    std_metrics = {
        metric: _to_python_number(np.std([fold[metric] for fold in fold_metrics], ddof=1))
        for metric in metric_names
    }

    return mean_metrics, std_metrics


def run_model(model, df, name, feature_set, cv):
    X = df.drop(columns=[TARGET])
    y = df[TARGET]

    fold_metrics = []
    audit_folds = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Se crea una copia independiente para que cada fold se entrene desde cero.
        fold_model = clone(model)
        fold_model.fit(X_train, y_train)
        preds = fold_model.predict(X_test).astype(int)

        metrics = compute_metrics(y_test, preds)
        metrics["fold"] = fold
        fold_metrics.append(metrics)

        # Predicciones sobre el fold reservado: cada persona recibe una sola
        # predicción de un modelo que no fue entrenado con esa persona.
        audit_fold = audit_context.iloc[test_idx].copy()
        audit_fold.insert(1, "model_id", name)
        audit_fold.insert(2, "feature_set", str(feature_set))
        audit_fold.insert(3, "fold", fold)
        audit_fold["score"] = preds
        audit_folds.append(audit_fold)

    # "fold" no se promedia; únicamente identifica cada partición.
    metrics_for_aggregation = [
        {metric: value for metric, value in fold.items() if metric != "fold"}
        for fold in fold_metrics
    ]
    mean_metrics, std_metrics = aggregate_metrics(metrics_for_aggregation)

    audit_predictions = pd.concat(audit_folds, ignore_index=True)
    audit_columns = [
        "entity_id",
        "model_id",
        "feature_set",
        "fold",
        "race",
        "sex",
        "age_cat",
        "label_value",
        "score",
    ]
    audit_predictions = audit_predictions.loc[:, audit_columns]

    result = {
        "model": name,
        "cross_validation": {
            "method": "StratifiedKFold",
            "n_splits": cv.n_splits,
            "shuffle": cv.shuffle,
            "random_state": cv.random_state,
        },
        "metrics": mean_metrics,
        "metrics_std": std_metrics,
        "fold_metrics": fold_metrics,
    }
    return result, audit_predictions


## 4. Ejecutar experimentos y generar predicciones out-of-fold


In [6]:
# Configuración común de la validación cruzada.
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = []
audit_outputs = {}


def run_and_store(model, dataset, model_name, feature_set):
    result, audit_predictions = run_model(model, dataset, model_name, feature_set, cv)
    results.append(result)
    audit_outputs[model_name] = audit_predictions


# Logistic Regression
lr = LogisticRegression(max_iter=2000)

# run_and_store(lr, df2, "LR_2_features", 2)
# run_and_store(lr, df7, "LR_7_features", 7)
run_and_store(lr, df8, "LR_8_features", 8)

# SVM
svm = SVC(kernel="rbf")

# run_and_store(svm, df2, "SVM_2_features", 2)
# run_and_store(svm, df7, "SVM_7_features", 7)

# XGBoost
# xgb = XGBClassifier(
#     n_estimators=200,
#     max_depth=6,
#     learning_rate=0.1,
#     random_state=42,
#     eval_metric="logloss",
# )

# run_and_store(xgb, df8, "XGB_8_features", 8)

# Promedio y desviación estándar de las métricas.
summary = pd.DataFrame(
    [
        {
            "model": result["model"],
            **result["metrics"],
            **{f"{metric}_std": value for metric, value in result["metrics_std"].items()},
        }
        for result in results
    ]
)

summary


,model,accuracy,precision,recall,f1_score,accuracy_std,precision_std,recall_std,f1_score_std
0,LR_8_features,0.680497,0.680384,0.562492,0.615561,0.019697,0.026373,0.030094,0.025738


## 5. Guardar resultados y archivos de auditoría 


In [ ]:
OUTPUT_FILE = BASE_DIR / "results_stratified_10fold.json"

# with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
#     json.dump(results, f, indent=2, ensure_ascii=False)

# Se crea un archivo independiente por modelo para ejecutar Aequitas directamente.
# score: predicción binaria del modelo; label_value: resultado real.
audit_files = {}
for model_name, audit_predictions in audit_outputs.items():
    audit_file = BASE_DIR / f"aequitas_audit_{model_name}.csv"
    audit_predictions.to_csv(audit_file, index=False)
    audit_files[model_name] = audit_file

# print("Resultados guardados en:", OUTPUT_FILE)
print("\nArchivos listos para Aequitas:")
for model_name, audit_file in audit_files.items():
    print(f"- {model_name}: {audit_file}")



Archivos listos para Aequitas:
- LR_8_features: data/processed/aequitas_audit_LR_8_features.csv

Cada CSV contiene: entity_id, grupos protegidos, label_value y score (predicciones out-of-fold).
